In [3]:
"""
generate.py — Get Ahead Hawks site generator
---------------------------------------------
Put this file in the same folder as your Excel and template.html.

Usage:
    python generate.py

Output:
    index.html  (push this to GitHub Pages)

Requirements:
    pip install openpyxl
"""

import re
from openpyxl import load_workbook

# ── CONFIG — update this line when you rename your file ──────────────
EXCEL_FILE    = '2026 Master Spread Copy1.xlsx'
TEMPLATE_FILE = 'template.html'
OUTPUT_FILE   = 'index.html'
# ─────────────────────────────────────────────────────────────────────

REAL_PITCHERS = {
    'Anthony Bubba', 'Michael Featherstone', 'Sean Stanley', 'Tyler Blumetti',
    'Justin Gouin', 'Sam Smith', 'Evan Gentile', 'Jeffrey Blake', 'Michael Greco',
    'Logan Huzi', 'Julian Rondon', 'Joe Dooley', 'Joseph Ferrucci', 'Peter Morrissey',
    'Mike DeRosa', 'Mike DeRosa ', 'AJ Formato', 'Logan Smalley',
}

POSITIONS = {
    'Nico DiGioia':    'OF',  'Alek Popovich':   'OF',  'Jackson Ciccone':  'OF',
    'JJ Romatzick':    'SS',  'Jeremy Mangiameli':'3B',  'Ryan Knight':      'OF',
    'Patrick Joseph':  '2B',  'Cameron Boardman': 'C',   'Tyler Shannon':    'IF/1B',
    'Josh Kaplan':     'IF',  'Peyton West':      'IF',  'Zaynn Godbee':     '1B',
    'Ben Schulz':      'OF',  'John Russell':     'C',   'Evin Curtiss':     'OF',
    'Jayden Gaither':  'OF',  'Alyis Godbee':     'OF',  'Sean Newman':      'IF',
    'Jon Melarczik':   'IF',
}

# ── Helpers ───────────────────────────────────────────────────────────

def sheet_rows(wb, sheet):
    ws = wb[sheet]
    data = list(ws.iter_rows(values_only=True))
    headers = data[0]
    return [dict(zip(headers, row)) for row in data[1:]
            if any(v is not None for v in row)]

def parse_num(v, default=None):
    if v is None: return default
    if isinstance(v, (int, float)): return float(v)
    try: return float(str(v).strip())
    except: return default

def parse_avgops(v):
    if not v: return None, None
    s = str(v).strip()
    if '/' in s:
        parts = s.split('/')
        try:
            a = float(parts[0]) if parts[0] else None
            o = float(parts[1]) if parts[1] else None
            return a, o
        except: pass
    return None, None

def js(v):
    """Convert a Python value to a JS literal string."""
    if v is None:
        return 'null'
    if isinstance(v, bool):
        return 'true' if v else 'false'
    if isinstance(v, (int, float)):
        if v != v:  # NaN check
            return 'null'
        return repr(v)
    if isinstance(v, str):
        escaped = (v.replace('\\', '\\\\')
                    .replace("'", "\\'")
                    .replace('\n', '\\n')
                    .replace('\r', ''))
        return f"'{escaped}'"
    if isinstance(v, dict):
        inner = ', '.join(f"'{k}':{js(val)}" for k, val in v.items())
        return '{' + inner + '}'
    if isinstance(v, list):
        return '[' + ', '.join(js(i) for i in v) + ']'
    return 'null'

# ── Parse Excel ───────────────────────────────────────────────────────

print(f'Reading {EXCEL_FILE}...')
wb   = load_workbook(EXCEL_FILE, read_only=True)
wb_d = load_workbook(EXCEL_FILE, data_only=True)

hitters_raw  = sheet_rows(wb, 'Hitters')
splits_h     = {r['Player']: r for r in sheet_rows(wb, 'Splits by Hitters') if r.get('Player')}
pitch_h      = {r['Player']: r for r in sheet_rows(wb, 'Pitch Type by Hitters') if r.get('Player')}
reports_raw  = {r['Player']: r for r in sheet_rows(wb, 'Hitter Reports') if r.get('Player')}
pitchers_raw = [r for r in sheet_rows(wb, 'Pitchers')
                if r.get('Player', '').strip() in {p.strip() for p in REAL_PITCHERS}]
pitch_p      = {r['Player'].strip(): r for r in sheet_rows(wb, 'Pitch Type by Pitcher') if r.get('Player')}
splits_p     = {r['Player'].strip(): r for r in sheet_rows(wb, 'Splits by Pitcher') if r.get('Player')}
pitch_usage  = {r['Player'].strip(): r for r in sheet_rows(wb, 'Pitch Usage') if r.get('Player')}

# Leverage scores (from data_only workbook so formulas are evaluated)
lev_rows = sheet_rows(wb_d, 'Leverage Score')
lev = {r['PLAYER'].strip(): r for r in lev_rows
       if r.get('PLAYER') and isinstance(r.get('LEV SCORE'), float)}

# Compute ranks in Python (avoids Excel RANK formula quirks)
scored_list = sorted(lev.items(), key=lambda x: -x[1]['LEV SCORE'])
lev_rank_map = {name: i + 1 for i, (name, _) in enumerate(scored_list)}

# ── Build hitters ─────────────────────────────────────────────────────

def pitch_entry(ph, ao_key, chase_key, whiff_key, swing_key):
    ao = ph.get(ao_key)
    if ao is None:
        return None
    a, o = parse_avgops(ao)
    return {
        'avgOps': str(ao),
        'opsVal': o or 0,
        'chase':  parse_num(ph.get(chase_key)),
        'whiff':  parse_num(ph.get(whiff_key)),
        'swing':  parse_num(ph.get(swing_key)),
    }

hitters_out = []
for h in hitters_raw:
    name = h.get('Player')
    if not name:
        continue

    sp  = splits_h.get(name, {})
    ph  = pitch_h.get(name, {})
    rep = reports_raw.get(name, {})

    pd = {}
    fb  = pitch_entry(ph, 'FB AVG/OPS',  'FB Chase',  'FB Whiff',  'Swing %')
    bb  = pitch_entry(ph, 'BB AVG/OPS',  'BB Chase',  'BB Whiff',  ' Swing %')
    ofs = pitch_entry(ph, 'OFS AVG/OPS', 'OFS Chase', 'OFS Whiff', 'Swing %')
    if fb:  pd['FB']  = fb
    if bb:  pd['BB']  = bb
    if ofs: pd['OFS'] = ofs

    hitters_out.append({
        'name':      name,
        'pos':       POSITIONS.get(name, '—'),
        'avg':       parse_num(h.get('AVG')),
        'ops':       parse_num(h.get('OPS')),
        'gp':        int(parse_num(h.get('GP')) or 0),
        'rhpAvg':    parse_num(sp.get('RHP AVG')),
        'rhpOps':    parse_num(sp.get('RHP OPS')),
        'lhpAvg':    parse_num(sp.get('LHP AVG')),
        'lhpOps':    parse_num(sp.get('LHP OPS')),
        'rhpChase':  parse_num(sp.get('RHP Chase')),
        'rhpWhiff':  parse_num(sp.get('RHP Whiff')),
        'lhpChase':  parse_num(sp.get('LHP Chase')),
        'lhpWhiff':  parse_num(sp.get('LHP Whiff')),
        'pitchData': pd if pd else None,
        'rl':   rep.get('R/L Matchup Notes'),
        'risp': rep.get('RISP Notes'),
        'cold': rep.get('Cold Zone Notes'),
        'hot':  rep.get('Hot Zone Notes'),
    })

# ── Build pitchers ────────────────────────────────────────────────────

def pitch_detail_entry(pp, pfx):
    ao = pp.get(f'{pfx} AVG/OPS')
    if ao is None:
        return None
    return {
        'avgOps': str(ao),
        'chase':  parse_num(pp.get(f'{pfx} Chase')),
        'whiff':  parse_num(pp.get(f'{pfx} Whiff')),
        'swing':  parse_num(pp.get(f'{pfx} Swing')),
        'strike': parse_num(pp.get(f'{pfx} Strike')),
    }

pitchers_out = []
seen = set()
for p in pitchers_raw:
    name = p.get('Player', '').strip()
    if not name or name in seen:
        continue
    seen.add(name)

    pu = pitch_usage.get(name, {})
    sp = splits_p.get(name, {})
    pp = pitch_p.get(name, {})
    lv = lev.get(name, {})

    details = {px: pitch_detail_entry(pp, px) for px in ['FB', 'SL', 'CB', 'CH']}
    details = {k: v for k, v in details.items() if v}

    lev_score = lv.get('LEV SCORE') if lv else None
    lev_rank  = lev_rank_map.get(name) if isinstance(lev_score, float) else None

    pitchers_out.append({
        'name':      name,
        'hand':      p.get('Throws', 'R'),
        'era':       parse_num(p.get('ERA')),
        'whip':      parse_num(p.get('WHIP')),
        'ip':        parse_num(p.get('IP')),
        'fbPct':     parse_num(pu.get('FB %')),
        'slPct':     parse_num(pu.get('SL%')),
        'cbPct':     parse_num(pu.get('CB%')),
        'chPct':     parse_num(pu.get('CH%')),
        'rhhAvgOps': str(sp.get('RHH AVG/OPS', '—')),
        'lhhAvgOps': str(sp.get('LHH AVG/OPS', '—')),
        'rhhChase':  parse_num(sp.get('RHH Chase')),
        'rhhWhiff':  parse_num(sp.get('RHH Whiff')),
        'lhhChase':  parse_num(sp.get('LHH Chase')),
        'lhhWhiff':  parse_num(sp.get('LHH Whiff')),
        'rhhStrike': parse_num(sp.get('RHH Strike')),
        'lhhStrike': parse_num(sp.get('LHH Strike')),
        'rhhKbb':    str(sp.get('RHH K:BB', '—')),
        'lhhKbb':    str(sp.get('LHH K:BB', '—')),
        'riscpOps':  str(pu.get('RISP AVG/OPS', '—')),
        'pitchDetails': details if details else None,
        'levScore':  round(lev_score, 3) if isinstance(lev_score, float) else None,
        'levRank':   lev_rank,
    })

# ── Build reports ─────────────────────────────────────────────────────

reports_out = [
    {
        'name': name,
        'rl':   rep.get('R/L Matchup Notes', ''),
        'risp': rep.get('RISP Notes', ''),
        'cold': rep.get('Cold Zone Notes', ''),
        'hot':  rep.get('Hot Zone Notes', ''),
    }
    for name, rep in reports_raw.items()
    if rep.get('R/L Matchup Notes')
]

# ── Build JS array strings ────────────────────────────────────────────

def obj_to_js(obj, indent='  '):
    lines = [f"{indent}'{k}': {js(v)}" for k, v in obj.items()]
    return '{\n' + ',\n'.join(lines) + '\n}'

h_js = 'const hitters2026 = [\n' + ',\n'.join(obj_to_js(h) for h in hitters_out) + '\n];'
p_js = 'const pitchers2026 = [\n' + ',\n'.join(obj_to_js(p) for p in pitchers_out) + '\n];'
r_js = 'const reports2026 = [\n' + ',\n'.join(obj_to_js(r) for r in reports_out) + '\n];'

# ── Inject into template ──────────────────────────────────────────────

print(f'Reading {TEMPLATE_FILE}...')
html = open(TEMPLATE_FILE, encoding='utf-8').read()

html = re.sub(r'const hitters2026 = \[\];\s*// \{\{HITTERS_2026\}\}',  h_js, html)
html = re.sub(r'const pitchers2026 = \[\];\s*// \{\{PITCHERS_2026\}\}', p_js, html)
html = re.sub(r'const reports2026 = \[\];\s*// \{\{REPORTS_2026\}\}',   r_js, html)

with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    f.write(html)

print(f'Done! {len(hitters_out)} hitters, {len(pitchers_out)} pitchers, {len(reports_out)} reports')
print(f'Output: {OUTPUT_FILE}')
print('Push index.html to GitHub Pages to go live.')


Reading 2026 Master Spread Copy1.xlsx...
Reading template.html...
Done! 20 hitters, 17 pitchers, 15 reports
Output: index.html
Push index.html to GitHub Pages to go live.
